In [ ]:
# import os
# import whisper
# from moviepy.editor import VideoFileClip
# import json

# # Step 1: Extract audio from video
# def extract_audio(video_path, audio_path):
#     print("Extracting audio...")
#     video = VideoFileClip(video_path)
#     video.audio.write_audiofile(audio_path, fps=16000)
#     print(f"Audio saved as: {audio_path}")

# # Step 2: Transcribe using Whisper
# def transcribe_audio_whisper(audio_path):
#     print("Loading Whisper model...")
#     model = whisper.load_model("large")  # use "medium" or "large" for better results

#     print("Transcribing audio...")
#     result = model.transcribe(audio_path, language="hi", verbose=True)

#     transcripts = []
#     for segment in result["segments"]:
#         transcripts.append({
#             "start": round(segment["start"], 2),
#             "end": round(segment["end"], 2),
#             "transcription": segment["text"].strip()
#         })

#     return transcripts

# # Step 3: Full pipeline
# def process_video(video_path):
#     audio_path = "extracted_audio.wav"
#     extract_audio(video_path, audio_path)
#     results = transcribe_audio_whisper(audio_path)
    
#     # Save output as JSON
#     with open("transcription_output.json", "w", encoding="utf-8") as f:
#         json.dump(results, f, ensure_ascii=False, indent=2)

#     print("Transcription saved to 'transcription_output.json'")
#     return results

# # Run the script
# if __name__ == "__main__":
#     video_file = "v3.mp4"
#     process_video(video_file)

In [ ]:
# import os
# import json
# from moviepy.editor import VideoFileClip
# import whisper

# def extract_audio(video_path, audio_path="extracted_audio.wav"):
#     print("Extracting audio...")
#     video = VideoFileClip(video_path)
#     video.audio.write_audiofile(audio_path, codec="pcm_s16le", fps=16000)  # Save as WAV
#     print(f"Audio saved as: {audio_path}")

# def transcribe_audio_whisper(audio_path):
#     print("Loading Whisper 'large' model (may take time)...")
#     model = whisper.load_model("large")  # High-quality model

#     print("Transcribing audio...")
#     result = model.transcribe(audio_path, language="hi", verbose=True)

#     transcripts = []
#     for segment in result["segments"]:
#         transcripts.append({
#             "start": segment["start"],
#             "end": segment["end"],
#             "text": segment["text"]
#         })

#     return transcripts

# def process_video(video_path):
#     audio_path = "extracted_audio.wav"
#     extract_audio(video_path, audio_path)
#     results = transcribe_audio_whisper(audio_path)

#     with open("transcription_output.json", "w", encoding="utf-8") as f:
#         json.dump(results, f, ensure_ascii=False, indent=4)
#     print("Transcription saved to: transcription_output.json")

# if __name__ == "__main__":
#     process_video("v1.mp4")


In [ ]:
# import os
# import json
# import moviepy
# from moviepy.editor import VideoFileClip
# from sarvamai import SarvamAI

# # Step 1: Extract audio from video
# def extract_audio(video_path, audio_path):
#     print("🎧 Extracting audio from video...")
#     video = VideoFileClip(video_path)
#     video.audio.write_audiofile(audio_path, fps=16000)
#     print(f"✅ Audio saved at: {audio_path}")


# # Step 2: Transcribe using Sarvam AI
# def transcribe_with_sarvam(audio_path):
#     print("🚀 Starting Sarvam transcription...")

#     client = SarvamAI(api_subscription_key="YOUR_API_KEY_HERE")

#     # Create transcription job
#     job = client.speech_to_text_job.create_job(
#         model="saaras:v3",
#         mode="transcribe",
#         language_code="unknown",
#         with_diarization=True,
#         num_speakers=1
#     )

#     # Upload file
#     job.upload_files(file_paths=[audio_path])

#     # Start job
#     job.start()
#     print("⏳ Waiting for job to complete...")
#     job.wait_until_complete()

#     # Get results
#     file_results = job.get_file_results()

#     print(f"\n✅ Successful: {len(file_results['successful'])}")
#     print(f"❌ Failed: {len(file_results['failed'])}")

#     # Download outputs
#     if file_results["successful"]:
#         output_dir = "./output"
#         job.download_outputs(output_dir=output_dir)
#         print(f"📂 Outputs downloaded to: {output_dir}")

#         return output_dir
#     else:
#         print("⚠️ No successful transcription.")
#         return None


# # Step 3: Full pipeline
# def process_video(video_path):
#     audio_path = "extracted_audio.wav"

#     # Extract audio
#     extract_audio(video_path, audio_path)

#     # Transcribe
#     output_dir = transcribe_with_sarvam(audio_path)

#     # Optional: Load JSON result
#     if output_dir:
#         files = os.listdir(output_dir)
#         json_files = [f for f in files if f.endswith(".json")]

#         if json_files:
#             result_path = os.path.join(output_dir, json_files[0])

#             with open(result_path, "r", encoding="utf-8") as f:
#                 data = json.load(f)

#             # Save clean copy
#             with open("transcription_output.json", "w", encoding="utf-8") as f:
#                 json.dump(data, f, indent=2, ensure_ascii=False)

#             print("📝 Final transcription saved as 'transcription_output.json'")
#             return data

#     return None


# # Run
# if __name__ == "__main__":
#     video_file = "v1.mp4"
#     process_video(video_file)

In [ ]:
import os
import json
import subprocess
from sarvamai import SarvamAI

# Step 1: Extract audio from video using FFmpeg
def extract_audio(video_path, audio_path):
    print("🎧 Extracting audio using FFmpeg...")

    command = [
        "ffmpeg",
        "-y",                 # overwrite if exists
        "-i", video_path,
        "-ar", "16000",       # sample rate
        "-ac", "1",           # mono audio
        audio_path
    ]

    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    if result.returncode != 0:
        print("❌ FFmpeg error:")
        print(result.stderr.decode())
        raise Exception("Audio extraction failed")

    print(f"✅ Audio saved at: {audio_path}")


# Step 2: Transcribe using Sarvam AI
def transcribe_with_sarvam(audio_path):
    print("🚀 Starting Sarvam transcription...")

    client = SarvamAI(api_subscription_key="dd")

    job = client.speech_to_text_job.create_job(
        model="saaras:v3",
        mode="transcribe",
        language_code="unknown",
        with_diarization=True,
        num_speakers=1
    )

    job.upload_files(file_paths=[audio_path])

    job.start()
    print("⏳ Waiting for job to complete...")
    job.wait_until_complete()

    file_results = job.get_file_results()

    print(f"\n✅ Successful: {len(file_results['successful'])}")
    print(f"❌ Failed: {len(file_results['failed'])}")

    if file_results["successful"]:
        output_dir = "./output"
        job.download_outputs(output_dir=output_dir)
        print(f"📂 Outputs downloaded to: {output_dir}")
        return output_dir
    else:
        print("⚠️ No successful transcription.")
        return None


# Step 3: Full pipeline
def process_video(video_path):
    audio_path = "extracted_audio.wav"

    extract_audio(video_path, audio_path)
    output_dir = transcribe_with_sarvam(audio_path)

    if output_dir:
        files = os.listdir(output_dir)
        json_files = [f for f in files if f.endswith(".json")]

        if json_files:
            result_path = os.path.join(output_dir, json_files[0])

            with open(result_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            with open("transcription_output.json", "w", encoding="utf-8") as f:
                json.dump(data, f, indent=2, ensure_ascii=False)

            print("📝 Final transcription saved as 'transcription_output.json'")
            return data

    return None


# Run
if __name__ == "__main__":
    video_file = "v1.mp4"
    process_video(video_file)

import json
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
# from langchain.chains.retrieval_qa.base import RetrievalQA


# Load embedding model
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


# Step 1: Load JSON
def load_transcript(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)


# Step 2: Convert to LangChain Documents
def create_documents(data):
    docs = []
    entries = data["diarized_transcript"]["entries"]

    for entry in entries:
        text = entry["transcript"]

        metadata = {
            "start": entry["start_time_seconds"],
            "end": entry["end_time_seconds"],
            "speaker": entry["speaker_id"]
        }

        docs.append(Document(page_content=text, metadata=metadata))

    return docs


# Step 3: Custom embedding function (for FAISS)
def embed_texts(texts):
    return embed_model.encode(texts, normalize_embeddings=True)


# Step 4: Create FAISS index
def create_vectorstore(docs):
    texts = [doc.page_content for doc in docs]
    embeddings = embed_texts(texts)

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(np.array(embeddings))

    return index, docs


# Step 5: Retriever
def retrieve(query, index, docs, k=3):
    query_embedding = embed_texts([query])
    scores, indices = index.search(np.array(query_embedding), k)

    return [docs[i] for i in indices[0]]


# Step 6: LLM (Groq - Llama 3)
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="dd"
)


# Prompt
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a helpful assistant.

Use the context below to answer the question.
Always include timestamps and speaker info.

Context:
{context}

Question:
{question}

Answer:
"""
)


# Step 7: QA
def answer_query(query, index, docs):
    retrieved_docs = retrieve(query, index, docs)

    context = "\n".join([
        f"[{d.metadata['start']:.2f}s - {d.metadata['end']:.2f}s] "
        f"(Speaker {d.metadata['speaker']}): {d.page_content}"
        for d in retrieved_docs
    ])

    final_prompt = prompt_template.format(
        context=context,
        question=query
    )

    response = llm.invoke(final_prompt)
    return response.content


# Full pipeline
def run_rag(json_path):
    data = load_transcript(json_path)
    docs = create_documents(data)

    index, docs = create_vectorstore(docs)

    while True:
        query = input("\nAsk a question (or 'exit'): ")
        if query.lower() == "exit":
            break

        answer = answer_query(query, index, docs)

        print("\n🧠 Answer:")
        print(answer)


# Run
if __name__ == "__main__":
    run_rag("transcription_output.json")

🎧 Extracting audio using FFmpeg...
✅ Audio saved at: extracted_audio.wav
🚀 Starting Sarvam transcription...
⏳ Waiting for job to complete...

✅ Successful: 1
❌ Failed: 0
📂 Outputs downloaded to: ./output
📝 Final transcription saved as 'transcription_output.json'


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Ask a question (or 'exit'):  exit
